In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import IPython.display as ipd
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, cross_val_score
from glob import glob
from itertools import cycle
import seaborn as sns
from pathlib import Path
from collections import Counter
import joblib

sns.set_theme(style="white", palette=None)

In [2]:
def cleanUp(audioInfo):
   X = [item["features"] for item in audioInfo]
   Y = [item["chord"] for item in audioInfo]
   X = np.array(X)
   Y = np.array(Y)
   # print(X.shape)
   # print(Y.shape)
   # print(Counter(Y))
   return X, Y


In [3]:
def extract_features(y, sr):
    target_duration = 5
    if len(y) > target_duration * sr:
        y_fixed = y[: target_duration * sr]
    else:
        padding = (target_duration * sr) - len(y)
        y_fixed = np.pad(y, (0, padding), mode="constant")

    y_harmonic = librosa.effects.harmonic(y_fixed)

    chroma = librosa.feature.chroma_cqt(y=y_harmonic, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)

    return np.concatenate([
        chroma.mean(axis=1), chroma.std(axis=1),
        tonnetz.mean(axis=1), tonnetz.std(axis=1),
    ])

In [4]:
# def create_dataset(audioFiles):
#     audioInfo = []
#     target_duration = 5
#     for file in audioFiles:
#         y, sr = librosa.load(file)
#         if(len(y) > target_duration * sr):
#             y_fixed = y[:target_duration * sr]
#         else:
#             padding = (target_duration * sr) - len(y)
#             y_fixed = np.pad(y, (0, padding), mode="constant")
#         S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
#         S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
#         chordName = Path(file).parent.name
#         audioInfo.append({"features": S_db_mel, "chord": chordName})
#     return cleanUp(audioInfo)

In [5]:
def create_dataset(audioFiles):
    audioInfo = []
    for file in audioFiles:
        y, sr = librosa.load(file)
        features = extract_features(y, sr)
        chordName = Path(file).parent.name
        audioInfo.append({"features": features, "chord": chordName})
    return cleanUp(audioInfo)

In [6]:
trainFiles = glob("./raw_dataset/Train/*/*.wav")
testFiles = glob("./raw_dataset/Test/*/*.wav")
X_train, Y_train = create_dataset(trainFiles)
X_test, Y_test = create_dataset(testFiles)

print("Training files found:", len(trainFiles))
print("Test files found:", len(testFiles))
print(Counter(Y_train))

Training files found: 630
Test files found: 142
Counter({np.str_('Am'): 40, np.str_('E'): 40, np.str_('A'): 25, np.str_('A#'): 25, np.str_('A#m'): 25, np.str_('B'): 25, np.str_('Bm'): 25, np.str_('C'): 25, np.str_('C#'): 25, np.str_('C#m'): 25, np.str_('Cm'): 25, np.str_('D'): 25, np.str_('D#'): 25, np.str_('D#m'): 25, np.str_('Dm'): 25, np.str_('Em'): 25, np.str_('F'): 25, np.str_('F#'): 25, np.str_('F#m'): 25, np.str_('Fm'): 25, np.str_('G'): 25, np.str_('G#'): 25, np.str_('G#m'): 25, np.str_('Gm'): 25})


In [7]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(Y_train)
y_test_encoded = encoder.transform(Y_test)

print("Classes:", encoder.classes_)
print("Feature vector length:", X_train.shape[1])

Classes: ['A' 'A#' 'A#m' 'Am' 'B' 'Bm' 'C' 'C#' 'C#m' 'Cm' 'D' 'D#' 'D#m' 'Dm' 'E'
 'Em' 'F' 'F#' 'F#m' 'Fm' 'G' 'G#' 'G#m' 'Gm']
Feature vector length: 36


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True)),
])

param_grid = {
    "svm__C": [1, 10, 50, 100],
    "svm__gamma": ["scale", 0.01, 0.001],
}

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(X_train, y_train_encoded)

model = grid.best_estimator_
print("Best params:", grid.best_params_)

accuracy = model.score(X_test, y_test_encoded)
print(f"Test accuracy: {accuracy:.2f}")

scores = cross_val_score(
    model,
    np.vstack([X_train, X_test]),
    np.concatenate([y_train_encoded, y_test_encoded]),
    cv=5,
)
print(f"Cross-val accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

/mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition/wsl2-env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition/wsl2-env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition/wsl2-env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=T

Best params: {'svm__C': 100, 'svm__gamma': 0.001}
Test accuracy: 0.80
Cross-val accuracy: 0.89 ± 0.04


/mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition/wsl2-env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [9]:
# joblib.dump(model, "chord_model.pkl")
# joblib.dump(encoder, "chord_encoder.pkl")

In [10]:
def testAudio(file):
    y, sr = librosa.load(file)
    features = extract_features(y, sr).reshape(1, -1)

    probs = model.predict_proba(features)[0]
    best_idx = np.argmax(probs)
    chord = encoder.classes_[best_idx]
    confidence = probs[best_idx]
    return chord, confidence

In [11]:
myPlays = glob("./MyPlays/*.wav")
for f in myPlays:
    print(f"{Path(f).name}: {testAudio(f)}")

A_acoustic_guitar_fender_fa_series_test_1.wav: (np.str_('A'), np.float64(0.29013244140013156))
A_acoustic_guitar_fender_fa_series_test_2.wav: (np.str_('A'), np.float64(0.5765935835559053))
CMajor.wav: (np.str_('Am'), np.float64(0.11614457303284766))
EMinor.wav: (np.str_('Em'), np.float64(0.10688445151932519))
GMajor.wav: (np.str_('G'), np.float64(0.21704535069346745))
testE_Minor.wav: (np.str_('D#'), np.float64(0.25392424587467066))
